# 📖 Notebook 3: Ledger & Double-Entry Bookkeeping

In a payment system, every dollar must be accounted for. If your numbers don't add up, you have a bug — or fraud.  
The solution used by every financial system since the 15th century is **double-entry bookkeeping**.

## The Core Rule

Every time money moves, we record **two** entries:
- A **debit** (money coming from somewhere)
- A **credit** (money going to somewhere)

The total of all debits must **always** equal the total of all credits. If they don't, something is wrong.

## Learning Objectives

By the end of this notebook you'll understand:
- Why double-entry bookkeeping exists and how it catches errors
- How to model ledger entries in a database
- How to verify the books balance (debits == credits)
- How to query account balances and transaction history

## 🛠️ Setup

```bash
cd system-designs/payment-system
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import uuid

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "payment_demo", "user": "demo", "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

conn = get_db()
print("✅ Postgres connected")
conn.close()

---
## 1. Why Double-Entry? A Simple Analogy

Think of it like moving a ball between two boxes:

- **Box A** (the customer's wallet) loses the ball → that's a **debit** on their account.
- **Box B** (the merchant's account) gains the ball → that's a **credit** on their account.

If you ever count all the balls and find the total has changed, something went wrong.  
Double-entry bookkeeping is just a formal way of making sure **no balls disappear or appear from nowhere**.

### Our Accounts

For a simplified payment system, we use two main accounts:

| Account | What It Represents |
|---------|-------------------|
| `customer_receivable` | Money we expect to collect from the customer's card |
| `merchant_payable` | Money we owe to the merchant |

When a charge succeeds:
- **Debit** `customer_receivable` (we received money from the customer)
- **Credit** `merchant_payable` (we owe that money to the merchant)

---
## 2. Examining the Existing Ledger

Our `init.sql` already seeded some ledger entries. Let's look at them.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Show all ledger entries
cur.execute("""
    SELECT le.transaction_id, le.account_name, le.entry_type, 
           le.amount_cents, le.description
    FROM ledger_entries le
    ORDER BY le.transaction_id, le.entry_type
""")

rows = cur.fetchall()
print(f"Total ledger entries: {len(rows)}\n")
print(f"{'Transaction':<15} {'Account':<25} {'Type':<8} {'Amount':>10}  Description")
print("-" * 90)
for r in rows:
    amt = f"${r['amount_cents']/100:.2f}"
    print(f"{r['transaction_id']:<15} {r['account_name']:<25} {r['entry_type']:<8} {amt:>10}  {r['description']}")

cur.close(); conn.close()

Notice the pattern: **every transaction has exactly two entries** — one debit and one credit for the same amount.

---
## 3. The Balance Check: Do the Books Balance?

The fundamental accounting equation:  
**Total Debits = Total Credits**

If this ever fails, we have a bug. Let's verify.

In [ ]:
def check_books_balance():
    """Verify that total debits equal total credits across all ledger entries."""
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""
        SELECT 
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits
        FROM ledger_entries
    """)
    row = cur.fetchone()
    total_debits = row[0] or 0
    total_credits = row[1] or 0

    print(f"Total Debits  : ${total_debits/100:.2f}")
    print(f"Total Credits : ${total_credits/100:.2f}")
    print(f"Difference    : ${(total_debits - total_credits)/100:.2f}")

    if total_debits == total_credits:
        print("\n✅ Books balance! Every dollar is accounted for.")
    else:
        print("\n🚨 BOOKS DO NOT BALANCE! Something is wrong!")

    cur.close(); conn.close()
    return total_debits == total_credits

check_books_balance()

---
## 4. Recording a New Charge with Ledger Entries

Let's build a function that processes a charge and records both ledger entries in a single database transaction.  
Using a database transaction (`BEGIN ... COMMIT`) ensures that either **both** entries are written, or **neither** is. We never end up with a debit without its matching credit.

In [ ]:
def record_charge_with_ledger(merchant_id, amount_cents, description):
    """
    Create a PaymentIntent, a Transaction, and the matching ledger entries.
    Everything happens in a single database transaction for atomicity.
    """
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        # Create PaymentIntent (succeeded immediately for this demo)
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
            VALUES (%s, %s, %s, 'usd', %s, 'succeeded')
        """, (pi_id, merchant_id, amount_cents, description))

        # Create Transaction
        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'charge', %s, 'usd', 'succeeded', '4242', 'visa')
        """, (txn_id, pi_id, amount_cents))

        # Double-entry ledger: DEBIT customer_receivable, CREDIT merchant_payable
        cur.execute("""
            INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
            VALUES
                (%s, 'customer_receivable', 'debit',  %s, 'usd', %s),
                (%s, 'merchant_payable',    'credit', %s, 'usd', %s)
        """, (txn_id, amount_cents, f"Charge for {pi_id}",
              txn_id, amount_cents, f"Charge for {pi_id}"))

        conn.commit()
        print(f"✅ Charge recorded:")
        print(f"   PaymentIntent : {pi_id}")
        print(f"   Transaction   : {txn_id}")
        print(f"   Amount        : ${amount_cents/100:.2f}")
        print(f"   Ledger        : DEBIT customer_receivable ${amount_cents/100:.2f}")
        print(f"                   CREDIT merchant_payable   ${amount_cents/100:.2f}")
        return txn_id

    except Exception as e:
        conn.rollback()
        print(f"❌ Error: {e}")
        return None

    finally:
        cur.close(); conn.close()

# Record a few charges
record_charge_with_ledger("merch_001", 5999, "Ledger demo: premium headphones")
print()
record_charge_with_ledger("merch_002", 1250, "Ledger demo: basic widget")

In [ ]:
# Verify the books still balance after our new charges
check_books_balance()

---
## 5. Account Balances

We can compute the balance of any account by summing its debits and credits.  
This is how you'd answer questions like "how much do we owe merchant X?" or "how much have we collected from customers?"

In [ ]:
def show_account_balances():
    """Show the balance of each account in the ledger."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT 
            account_name,
            SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS total_debits,
            SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS total_credits,
            COUNT(*) AS entry_count
        FROM ledger_entries
        GROUP BY account_name
        ORDER BY account_name
    """)

    print(f"{'Account':<25} {'Debits':>12} {'Credits':>12} {'Net':>12}  Entries")
    print("-" * 75)
    for r in cur.fetchall():
        net = r['total_debits'] - r['total_credits']
        print(f"{r['account_name']:<25} ${r['total_debits']/100:>10.2f} ${r['total_credits']/100:>10.2f} ${net/100:>10.2f}  {r['entry_count']}")

    cur.close(); conn.close()

show_account_balances()

---
## 6. What Happens When the Books Don't Balance?

Let's intentionally break the ledger by inserting a single entry without its matching pair.  
This simulates a bug where the credit entry wasn't written (e.g., the app crashed between the two inserts).

In [ ]:
conn = get_db()
cur = conn.cursor()

# Insert ONLY a debit — no matching credit (this is the bug!)
cur.execute("""
    INSERT INTO ledger_entries (transaction_id, account_name, entry_type, amount_cents, currency, description)
    VALUES ('txn_BUG', 'customer_receivable', 'debit', 9999, 'usd', 'BUG: missing credit entry')
""")
conn.commit()
cur.close(); conn.close()

print("💣 Inserted a debit without a matching credit...")
print()
check_books_balance()

In [ ]:
# Find the broken entry
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Find transactions where debits != credits
cur.execute("""
    SELECT 
        transaction_id,
        SUM(CASE WHEN entry_type = 'debit'  THEN amount_cents ELSE 0 END) AS debits,
        SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END) AS credits
    FROM ledger_entries
    GROUP BY transaction_id
    HAVING SUM(CASE WHEN entry_type = 'debit' THEN amount_cents ELSE 0 END)
        != SUM(CASE WHEN entry_type = 'credit' THEN amount_cents ELSE 0 END)
""")

broken = cur.fetchall()
print("🔍 Transactions where debits ≠ credits:")
for r in broken:
    print(f"  {r['transaction_id']}: debits=${r['debits']/100:.2f}  credits=${r['credits']/100:.2f}")

cur.close(); conn.close()

In [ ]:
# Clean up the broken entry
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM ledger_entries WHERE transaction_id = 'txn_BUG'")
conn.commit()
cur.close(); conn.close()

print("🧹 Cleaned up the broken entry.")
check_books_balance()

---
## 7. Per-Merchant Balances

In a real system, you'd want to see how much each merchant has earned.  
We can join the ledger with transactions and payment intents to break down balances by merchant.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT 
        m.name AS merchant_name,
        COUNT(DISTINCT le.transaction_id) AS transaction_count,
        SUM(CASE WHEN le.entry_type = 'credit' AND le.account_name = 'merchant_payable' 
            THEN le.amount_cents ELSE 0 END) AS total_earned_cents
    FROM ledger_entries le
    JOIN transactions t ON le.transaction_id = t.id
    JOIN payment_intents pi ON t.payment_intent_id = pi.id
    JOIN merchants m ON pi.merchant_id = m.id
    GROUP BY m.name
    ORDER BY total_earned_cents DESC
""")

print(f"{'Merchant':<25} {'Transactions':>15} {'Total Earned':>15}")
print("-" * 60)
for r in cur.fetchall():
    print(f"{r['merchant_name']:<25} {r['transaction_count']:>15} ${r['total_earned_cents']/100:>13.2f}")

cur.close(); conn.close()

---
## 8. Summary

| Concept | Key Point |
|---------|----------|
| **Double-Entry** | Every money movement creates two entries: a debit and a credit |
| **Balance Check** | Total debits must always equal total credits |
| **Atomicity** | Both entries are written in a single DB transaction — all or nothing |
| **Auditability** | The ledger is append-only — you never update or delete entries |
| **Error Detection** | If debits ≠ credits for any transaction, you've found a bug |

### Why This Matters in System Design Interviews

When an interviewer asks "how do you ensure financial integrity?", the answer is:
1. **Double-entry bookkeeping** to track every dollar.
2. **Database transactions** to ensure atomic writes.
3. **Balance verification queries** to catch bugs.
4. **Append-only audit logs** for compliance.

➡️  Next notebook: **Fraud Detection Basics**